# 11.18 指定哪一塊文字，如何只讀那一塊？


這張字卡左邊寫「入口」，右邊寫「出口」。如果問題是「只讀右側框內的字」，希望答案是「出口」，不需要把左邊一起抄出來。先看清兩區的內容，再看綠色框指定了哪一區。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/new-11.18-two-regions.svg")))

字卡是作者製作的可見材料。「入口」和「出口」都只用到主線有限字表裡的字，這一題也不用先會讀任意街景招牌。框線提供位置，不提供字詞答案：把右框裡的內容換成「開關」，同一條「只讀右側框內的字」就應得到「開關」。

這個工作可以拆成兩個理解步驟。先選出使用者指定的區域，再辨認該區的字。選區回答「我要看哪裡」，讀字回答「那裡寫了什麼」。假如選到左區，卻完整讀出「入口」，辨字可能沒錯，指定範圍仍然錯了。假如選到右區，卻寫成「出入」，選區正確，字詞內容或次序仍然錯了。

最容易看懂的入口，是讓使用者指定框，再由固定的裁切步驟取出框內像素，送給自行訓練的讀字入口。裁切只按座標拿出一塊圖，不會認字；讀字入口仍要從像素取得特徵，再交給文字核心生成字串。這條路把區域選取交給已知的框，讓我們先專心教少量字形和字序。

也可以保留整張圖和框的位置，讓模型學會只使用指定區域。兩種安排的差別在於：前者直接移除框外像素，後者還要學會把注意放在框內。本節先使用指定框的安排，不把固定位置裁切說成「會自動找到任何圖片裡的文字」。自動尋找文字位置是另一個任務，需要相應的材料與檢查。

訓練示範應把整張材料、指定框和正確字串連在一起。可以先用已知字表的單字與短詞，讓模型學字形對應，再增加少量字的組合。框內「出口」的示範答案是「出口」；若總把左右文字一起寫進答案，模型學到的就會是整張抄寫。答案範圍必須和要求一致。

可以用同一張圖安排三個小對照。指定右框時，檢查是否只得到「出口」；改指定左框時，檢查是否只得到「入口」；保持右框不動，但把兩區內容互換時，檢查是否改讀「入口」。最後一個對照很重要：如果右側位置永遠放「出口」，模型可能只靠位置猜詞，未必使用字形。

這些對照也讓錯誤更容易解讀。「入口出口」包含正確字詞，卻多抄了框外內容；「出口」符合內容與範圍；「口出」用了相同兩個字，次序仍不符合原圖。前面已學過逐字與字序的檢查，這裡多加的是選取範圍，而不是重新發明一種讀字分數。

驗收有限字卡時，還要留出字體或版面變體，並試已知字元組成的新短串。這樣才知道模型是否只記住一張「出口」圖。讀過這張清晰字卡，也不足以推出它能閱讀模糊、傾斜或未見字的真實招牌。

<details>
<summary>字型來源與前文連結</summary>

字卡短詞與希望答案由作者選寫，是教材設計例；SVG 指定 Noto Sans CJK TC 並提供系統中文字型作備援。Noto CJK 的[官方字型授權](https://github.com/notofonts/noto-cjk/blob/main/Sans/LICENSE)是 SIL OFL 1.1。使用字型產生文件與重新散佈字型檔，是不同的授權情況；本節沒有打包字型檔。

[11.15](https://birdhackor.github.io/tiny-perceptron-vlm/11.15.html)說明筆畫資訊，[11.16](https://birdhackor.github.io/tiny-perceptron-vlm/11.16.html)說明逐字與順序；本節再問指定框是否改變應讀的內容。

</details>
